# Validate Market Data

This notebook validates the downloaded stock price data and NASDAQ index data before return calculation and event-study analysis.

It performs the following checks:

- verifies that the raw stock price file and NASDAQ index file exist
- verifies that required columns are present
- parses and validates date fields
- checks for duplicate rows
- checks for missing values in key columns
- verifies ticker coverage against the event metadata
- verifies that each event trading day has matching stock and index data
- summarizes the date coverage of the downloaded data

This notebook does **not** modify the raw datasets. It only verifies that the market data download pipeline output is correct.

In [1]:
import pandas as pd
from pathlib import Path

## 1. Define file paths

We first define the file paths used in the validation process.

In [2]:
metadata_path = Path("../data/processed/event_metadata_final.csv")
prices_path = Path("../data/raw/market_data/prices_raw.csv")
index_path = Path("../data/raw/market_data/nasdaq_index_raw.csv")

## 2. Check that required files exist

This step verifies that the metadata file, stock price file, and NASDAQ index file are present before loading them.

In [3]:
required_files = {
    "event metadata": metadata_path,
    "stock prices": prices_path,
    "NASDAQ index": index_path,
}

for name, path in required_files.items():
    print(f"{name}: {path.exists()} -> {path}")
    assert path.exists(), f"Missing required file: {path}"

event metadata: True -> ../data/processed/event_metadata_final.csv
stock prices: True -> ../data/raw/market_data/prices_raw.csv
NASDAQ index: True -> ../data/raw/market_data/nasdaq_index_raw.csv


## 3. Load datasets

We now load the event metadata, stock price data, and NASDAQ index data into pandas DataFrames.

In [4]:
metadata = pd.read_csv(metadata_path)
prices = pd.read_csv(prices_path)
index_df = pd.read_csv(index_path)

print("metadata shape:", metadata.shape)
print("prices shape:", prices.shape)
print("index shape:", index_df.shape)

metadata shape: (188, 15)
prices shape: (13170, 8)
index shape: (1317, 7)


## 4. Preview the datasets

A quick preview helps confirm that the files were loaded correctly and have the expected structure.

In [5]:
display(metadata.head())
display(prices.head())
display(index_df.head())

,ticker,file_name,file_path,call_date_from_filename,year,quarter_label,call_datetime_header_raw,call_datetime_parsed,call_datetime_gmt,call_datetime_et,call_time_gmt,call_time_et,after_market_close_et,event_trading_day_final,error
0,AAPL,2016-Jan-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-01-26,2016,Q1,"JANUARY 26, 2016 / 10:00PM GMT",2016-01-26T22:00:00+00:00,2016-01-26T22:00:00+00:00,2016-01-26T17:00:00-05:00,22:00:00,17:00:00,True,2016-01-27,NaN
1,AAPL,2016-Apr-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-04-26,2016,Q2,"APRIL 26, 2016 / 9:00PM GMT",2016-04-26T21:00:00+00:00,2016-04-26T21:00:00+00:00,2016-04-26T17:00:00-04:00,21:00:00,17:00:00,True,2016-04-27,NaN
2,AAPL,2016-Jul-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-07-26,2016,Q3,"JULY 26, 2016 / 9:00PM GMT",2016-07-26T21:00:00+00:00,2016-07-26T21:00:00+00:00,2016-07-26T17:00:00-04:00,21:00:00,17:00:00,True,2016-07-27,NaN
3,AAPL,2016-Oct-25-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-10-25,2016,Q4,"OCTOBER 25, 2016 / 9:00PM GMT",2016-10-25T21:00:00+00:00,2016-10-25T21:00:00+00:00,2016-10-25T17:00:00-04:00,21:00:00,17:00:00,True,2016-10-26,NaN
4,AAPL,2017-Jan-31-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2017-01-31,2017,Q1,"JANUARY 31, 2017 / 10:00PM GMT",2017-01-31T22:00:00+00:00,2017-01-31T22:00:00+00:00,2017-01-31T17:00:00-05:00,22:00:00,17:00:00,True,2017-02-01,NaN


,Date,ticker,Open,High,Low,Close,Adj Close,Volume
0,2015-06-29,AAPL,31.365000,31.617500,31.120001,31.132500,27.805971,196645600
1,2015-06-30,AAPL,31.392500,31.530001,31.215000,31.357500,28.006931,177482800
2,2015-07-01,AAPL,31.725000,31.735001,31.497499,31.650000,28.268185,120955200
3,2015-07-02,AAPL,31.607500,31.672501,31.442499,31.610001,28.232452,108844000
4,2015-07-06,AAPL,31.235001,31.557501,31.212500,31.500000,28.134211,112241600


,Date,Adj Close,Close,High,Low,Open,Volume
0,2015-06-29,4958.470215,4958.470215,5051.009766,4956.229980,5021.209961,2025580000
1,2015-06-30,4986.870117,4986.870117,5008.759766,4968.259766,5000.149902,2034430000
2,2015-07-01,5013.120117,5013.120117,5038.549805,4994.459961,5029.049805,1814560000
3,2015-07-02,5009.209961,5009.209961,5027.470215,4990.740234,5024.299805,1490810000
4,2015-07-06,4991.939941,4991.939941,5020.709961,4960.930176,4963.799805,1741500000


## 5. Validate required columns

This step checks that each dataset contains the required columns needed for the event study pipeline.